# Document Question Answering System (RAG)

Pipeline: ingest document -> chunk -> embed (pretrained model) -> store in a vector database -> retrieve relevant chunks for a question -> build a prompt with context + question -> generate a grounded answer with a language model.

- **Embeddings:** sentence-transformers all-MiniLM-L6-v2 (384-dim, pretrained)
- **Vector database:** FAISS (IndexFlatIP, cosine similarity via normalized vectors)
- **Retrieval:** hybrid — vector similarity combined with a TF-IDF keyword score
- **LLM:** google/flan-t5-base (local, no API key needed)



## Install dependencies

Run this once. Skip it if you already have these installed.

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers torch pypdf datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.9 MB/s eta 0:00:00


## 1. Imports

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss

## 2. Document ingestion

Accepts a plain text file, a PDF (extracted with pypdf), or a Hugging Face dataset (via datasets).

In [ ]:
def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text


def load_hf_dataset(dataset_name, text_column="text", split="train", limit=200):
    """Loads a Hugging Face dataset and joins the chosen text column into one
    document string (limit caps how many rows are pulled in, for speed)."""
    from datasets import load_dataset
    ds = load_dataset(dataset_name, split=split)
    rows = ds[text_column][:limit]
    return "\n".join(rows)


def load_document(path):
    """Dispatches to the right loader based on file extension."""
    if path.lower().endswith(".pdf"):
        return load_pdf(path)
    return load_txt(path)

## 3. Chunking

Sentence-aware chunking: sentences are grouped together until a word budget is reached, so a chunk never ends in the middle of a sentence. A small overlap (1 sentence) is kept between consecutive chunks so context isn't lost at the edges.

In [ ]:
def split_sentences(text):
    text = re.sub(r'([^.!?\s])\n', r'\1. ', text)  # newline after non-punct = sentence break
    text = text.replace("\n", " ")
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip() and s.strip() != "."]


def chunk_text(text, chunk_size=150, overlap=1):
    sentences = split_sentences(text)
    chunks = []
    current, current_len = [], 0

    for sentence in sentences:
        sentence_len = len(sentence.split())
        if current_len + sentence_len > chunk_size and current:
            chunks.append(" ".join(current))
            current = current[-overlap:] if overlap else []
            current_len = sum(len(s.split()) for s in current)
        current.append(sentence)
        current_len += sentence_len

    if current:
        chunks.append(" ".join(current))
    return chunks

## 4 & 5. Embeddings + vector database

Embedder wraps the pretrained all-MiniLM-L6-v2 model (falls back to TF-IDF if it can't be downloaded). VectorStore builds a FAISS index over the normalized embeddings, so inner product = cosine similarity — this is the actual vector database step.

In [ ]:
class Embedder:
    def __init__(self):
        self.mode = "transformer"
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")
        except Exception as e:
            print(f"[warning] could not load sentence-transformers model ({e}). "
                  f"Falling back to TF-IDF embeddings.")
            self.mode = "tfidf"
            self.tfidf = TfidfVectorizer(stop_words="english")

    def fit_transform(self, texts):
        if self.mode == "transformer":
            vectors = self.model.encode(texts, convert_to_numpy=True)
            return vectors.astype("float32")
        else:
            vectors = self.tfidf.fit_transform(texts).toarray()
            return vectors.astype("float32")

    def transform(self, texts):
        if self.mode == "transformer":
            vectors = self.model.encode(texts, convert_to_numpy=True)
            return vectors.astype("float32")
        else:
            vectors = self.tfidf.transform(texts).toarray()
            return vectors.astype("float32")


class VectorStore:
    def __init__(self, chunks, embedder):
        self.chunks = chunks
        self.embedder = embedder
        vectors = embedder.fit_transform(chunks)
        faiss.normalize_L2(vectors)
        self.vectors = vectors
        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)

    def query(self, question, top_k=3):
        q_vector = self.embedder.transform([question])
        faiss.normalize_L2(q_vector)
        scores, indices = self.index.search(q_vector, top_k)
        return [(self.chunks[i], float(scores[0][pos]))
                for pos, i in enumerate(indices[0]) if i != -1]

## 6. Hybrid retrieval

Combines the vector store's similarity score with a plain TF-IDF keyword score, so an exact term match (e.g. "k-fold") gets a boost even if the semantic embedding ranks it lower.

In [ ]:
class HybridRetriever:
    def __init__(self, chunks, vector_store, alpha=0.7):
        self.chunks = chunks
        self.vector_store = vector_store
        self.alpha = alpha  # weight given to the vector score vs keyword score
        self.keyword_vectorizer = TfidfVectorizer(stop_words="english")
        self.keyword_vectors = self.keyword_vectorizer.fit_transform(chunks)

    def retrieve(self, question, top_k=3, pool=6):
        vector_hits = self.vector_store.query(question, top_k=pool)
        q_keyword_vector = self.keyword_vectorizer.transform([question])

        scored = []
        for chunk, vector_score in vector_hits:
            idx = self.chunks.index(chunk)
            keyword_score = (self.keyword_vectors[idx] @ q_keyword_vector.T).toarray()[0][0]
            combined = self.alpha * vector_score + (1 - self.alpha) * keyword_score
            scored.append((chunk, combined, vector_score, keyword_score))

        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

## 7. Prompt construction + LLM answer generation

Builds a prompt from the retrieved context + question, and feeds it to flan-t5-base (falls back to picking the most relevant sentences from the context if the model can't be downloaded).

In [ ]:
class AnswerGenerator:
    def __init__(self):
        self.mode = "llm"
        try:
            from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
            model_name = "google/flan-t5-base"
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        except Exception as e:
            print(f"[warning] could not load flan-t5-base ({e}). "
                  f"Falling back to extractive answers.")
            self.mode = "extractive"

    def build_prompt(self, question, context_chunks):
        context = "\n\n".join(chunk for chunk, *_ in context_chunks)
        prompt = (
            "Answer the question using only the context below. "
            "If the answer is not in the context, say you don't know.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n"
            "Answer:"
        )
        return prompt

    def generate(self, question, context_chunks):
        if self.mode == "llm":
            prompt = self.build_prompt(question, context_chunks)
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            output_ids = self.model.generate(**inputs, max_new_tokens=100)
            return self.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        else:
            return self._extractive_fallback(question, context_chunks)

    def _extractive_fallback(self, question, context_chunks, top_sentences=2):
        all_sentences, seen = [], set()
        for chunk, *_ in context_chunks:
            for sentence in split_sentences(chunk):
                if sentence not in seen and len(sentence.split()) >= 6:
                    seen.add(sentence)
                    all_sentences.append(sentence)
        if not all_sentences:
            return "No relevant information found in the document."

        vectorizer = TfidfVectorizer(stop_words="english")
        sent_vectors = vectorizer.fit_transform(all_sentences + [question])
        q_vector = sent_vectors[-1]
        scores = (sent_vectors[:-1] @ q_vector.T).toarray().ravel()
        best_idx = sorted(np.argsort(scores)[::-1][:top_sentences])
        return " ".join(all_sentences[i] for i in best_idx)

## 8. Full pipeline

In [ ]:
def build_pipeline(document_path, chunk_size=150, overlap=1, alpha=0.7):
    document_text = load_document(document_path)
    chunks = chunk_text(document_text, chunk_size=chunk_size, overlap=overlap)

    embedder = Embedder()
    vector_store = VectorStore(chunks, embedder)
    retriever = HybridRetriever(chunks, vector_store, alpha=alpha)
    generator = AnswerGenerator()

    return chunks, retriever, generator


def answer_question(question, retriever, generator, top_k=3):
    retrieved = retriever.retrieve(question, top_k=top_k)
    answer = generator.generate(question, retrieved)
    return answer, retrieved

## 9. Build the pipeline on the sample document

In [ ]:
chunks, retriever, generator = build_pipeline("sample_document.txt")
print(f"Document split into {len(chunks)} chunks")
print(f"Embedding mode: {retriever.vector_store.embedder.mode}")
print(f"Generation mode: {generator.mode}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Document split into 6 chunks
Embedding mode: transformer
Generation mode: llm


## 10. Test with sample questions

In [ ]:
test_questions = [
    "What is the difference between supervised and unsupervised learning?",
    "What causes overfitting and how can it be reduced?",
    "What is gradient descent used for?",
    "What is k-fold cross validation?",
    "What is reinforcement learning used for?",
]

for q in test_questions:
    answer, retrieved = answer_question(q, retriever, generator)
    print("Q:", q)
    print("A:", answer)
    print("Top combined scores:", [round(float(s[1]), 3) for s in retrieved])
    print("-" * 80)

Q: What is the difference between supervised and unsupervised learning?
A: find hidden patterns in the data
Top combined scores: [0.516, 0.478, 0.202]
--------------------------------------------------------------------------------
Q: What causes overfitting and how can it be reduced?
A: Cross Validation.
Top combined scores: [0.387, 0.25, 0.202]
--------------------------------------------------------------------------------
Q: What is gradient descent used for?
A: an optimization algorithm used to minimize the loss function of a model
Top combined scores: [0.621, 0.278, 0.231]
--------------------------------------------------------------------------------
Q: What is k-fold cross validation?
A: supervised learning
Top combined scores: [0.557, 0.337, 0.176]
--------------------------------------------------------------------------------
Q: What is reinforcement learning used for?
A: find hidden patterns or groupings in the data without being told what the correct answer is.
Top combin

## 11. Try your own question

In [ ]:
my_question = "What is feature engineering?"
answer, retrieved = answer_question(my_question, retriever, generator)
print("Q:", my_question)
print("A:", answer)

Q: What is feature engineering?
A: is a branch of artificial intelligence that allows computer systems to learn patterns from data instead of being explicitly programmed with rules.
